## Topological feature extraction by persistent homology
This code includes

- Loading of images
- Calculation of persistent homology for patch images
- Generation of persistence image from persistent homology calculation results

The resulting feature vectors will be used in the evaluation code.

In [ ]:
from persim import PersistenceImager
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import homcloud.interface as hc
import tqdm
from scipy import ndimage
from concurrent.futures import ProcessPoolExecutor
import joblib
import cv2
import pandas as pd
import concurrent.futures

In [ ]:
image_df=pd.read_excel('MYDATA.xlsx',sheet_name='TCGA')

In [ ]:
### image paths
img_list=np.array(image_df['img_list'])
### histological classification
case_info=np.array(image_df['JPN_classification_patch'])

### Image loading function

In [ ]:
def load_and_process_image(i):
    img_path = img_list[i]
    img = cv2.imread(img_path, cv2.IMREAD_UNCHANGED)
    if img is not None:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (256, 256))
        return (img, case_info[i])
    else:
        return None

In [ ]:
image_list = []
sample_label = []

with concurrent.futures.ThreadPoolExecutor() as executor:
    results = list(tqdm.tqdm(executor.map(load_and_process_image, range(len(img_list))), total=len(img_list)))

for result in results:
    if result is not None:
        img, label = result
        image_list.append(img)
        sample_label.append(label)

100%|██████████| 6580/6580 [00:41<00:00, 157.84it/s]


### Persistent homology calculation and persistence image generation

We calculate PD0 and PD1 persistent homology with grayscale filtration for multiple eroded/dilated images.

Input image is a grayscale image with intensity values 0-255.

For persistence image generation, we use the following:

- matrix_size: resulting matrix size of persistence image
- sigma: gaussian distribution sigma for persistence image
- mid_birth: mean of births in persistence diagram. This will be the vertical center line for persistence image
- max_range: size of square for persistence image ROI. Here we set it by 6*max(std(persistence),std(birth))

In [ ]:
### Patch to persimg
matrix_size = 10
sigma = 10
def compute_persistence_image(img):
  pd0 = hc.PDList.from_bitmap_levelset(img).dth_diagram(0)
  pd1= hc.PDList.from_bitmap_levelset(img).dth_diagram(1)
  ### PD0 persimg calculation
  pairs = np.vstack([pd0.births, pd0.deaths]).T
  if len(pairs[:,0])==0:
    print("empty pair")
    zeroth_persimg=np.zeros((10,10))
  else:
    mid_birth = int(np.mean(pairs[:, 0]))
    max_range = (int(np.max((6 * np.std(pairs[:, 1] - pairs[:, 0]), 6 * np.std(pairs[:, 0]))) / 10) + 1) * 10
    pimgr = PersistenceImager(
        pixel_size=max_range / matrix_size,
        birth_range=(mid_birth - max_range / 2, mid_birth + max_range / 2),
        pers_range=(0, max_range),
        kernel_params={'sigma': sigma}
    )
    zeroth_persimg=pimgr.transform([pairs], skew=True)[0]
  ### PD1 persimg calculation
  pairs = np.vstack([pd1.births, pd1.deaths]).T
  if len(pairs[:,0])==0:
    print("empty pair")
    first_persimg=np.zeros((10,10))
  else:
    mid_birth = int(np.mean(pairs[:, 0]))
    max_range = (int(np.max((6 * np.std(pairs[:, 1] - pairs[:, 0]), 6 * np.std(pairs[:, 0]))) / 10) + 1) * 10
    pimgr = PersistenceImager(
        pixel_size=max_range / matrix_size,
        birth_range=(mid_birth - max_range / 2, mid_birth + max_range / 2),
        pers_range=(0, max_range),
        kernel_params={'sigma': sigma}
    )
    first_persimg=pimgr.transform([pairs], skew=True)[0]
  return zeroth_persimg, first_persimg

def process_image(image):### erosion/dilation and PH calculation with function above
  img = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
  # Precompute dilated and eroded images
  dilations = [img.copy()]
  erosions = [img.copy()]
  for i in range(1, 9):
    dilations.append(ndimage.grey_dilation(dilations[-1], size=(3, 3)))
    erosions.append(ndimage.grey_erosion(erosions[-1], size=(3, 3)))

  images_to_process = [
      img, dilations[2], dilations[4], dilations[6], dilations[8],
      erosions[2], erosions[4], erosions[6], erosions[8]
  ]

  pimgs_all = []
  for im in images_to_process:
    zeroth_pimg,first_pimg=compute_persistence_image(im)
    pimgs_all.append(zeroth_pimg)
    pimgs_all.append(first_pimg)

  return pimgs_all

In [7]:
with ProcessPoolExecutor() as executor:
    results = list(tqdm.tqdm(executor.map(process_image, image_list), total=6580))
pimgs_list = results

100%|██████████| 6580/6580 [19:08<00:00,  5.73it/s]


### We get 18 10*10 persistence images. In evaluation part, we flatten this result to get 1800 dimensional vector.

In [8]:
np.array(pimgs_list).shape

(6580, 18, 10, 10)

In [ ]:
#np.save('pimgs_list20260315.npy',np.array(pimgs_list))